In [ ]:
import os, re, glob, time
import numpy as np, pandas as pd, pydicom, cv2, torch
import torch.nn as nn, timm

BASE = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
dev  = "cuda" if torch.cuda.is_available() else "cpu"
T0, TIME_BUDGET = time.time(), 8.0 * 3600      # ৯ ঘণ্টার সীমা, ৮ এ থামব

N_SLICES, SIZE, PLANES = 12, 224, ["Sagittal", "Coronal", "Axial"]

sub = pd.read_csv(f"{BASE}/sample_submission.csv")
TARGETS = [c for c in sub.columns if c != "StudyInstanceUID"]
ts = pd.read_csv(f"{BASE}/test_series.csv")
print("টেস্ট স্টাডি:", len(sub), "| সিরিজ:", len(ts))

# ---------- cache builder-এর হুবহু একই লজিক ----------
def lat_from_header(d):
    v = str(getattr(d, "Laterality", "") or "").strip().upper()
    if v in ("L", "R"): return v
    desc = str(getattr(d, "SeriesDescription", "") or "").upper()
    if re.search(r"(RIGHT|\bRT\b|_R_|\bR\b)", desc): return "R"
    if re.search(r"(LEFT|\bLT\b|_L_|\bL\b)", desc):  return "L"
    return None

def pick_series(rows):
    out = {}
    for p in PLANES:
        s = rows[rows["Anatomical_Plane"] == p]
        out[p] = None if len(s)==0 else s.sort_values("Fluid_Sensitive", ascending=False)["SeriesInstanceUID"].iloc[0]
    return out

def load_block(folder):
    files = glob.glob(f"{folder}/*")
    if not files: return None, None
    order = []
    for f in files:
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            order.append((float(getattr(h, "InstanceNumber", 0) or 0), f, h))
        except Exception: pass
    if not order: return None, None
    order.sort(key=lambda x: x[0])
    lat = lat_from_header(order[0][2])

    n = len(order)
    lo, hi = int(n*0.2), max(int(n*0.8), int(n*0.2)+1)
    mid = order[lo:hi] or order
    idx = np.linspace(0, len(mid)-1, N_SLICES).round().astype(int)

    blk = np.zeros((N_SLICES, SIZE, SIZE), np.uint8)
    for k, i in enumerate(idx):
        try:
            a = pydicom.dcmread(mid[i][1]).pixel_array.astype(np.float32)
            p1, p99 = np.percentile(a, [1, 99])
            a = np.clip((a - p1) / (p99 - p1 + 1e-6), 0, 1)
            blk[k] = (cv2.resize(a, (SIZE, SIZE), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
        except Exception: pass
    return blk, lat

def build_study(uid):
    rows = ts[ts["StudyInstanceUID"] == uid]
    chosen = pick_series(rows)
    vol, lat = np.zeros((len(PLANES), N_SLICES, SIZE, SIZE), np.uint8), None
    for pi, p in enumerate(PLANES):
        if chosen[p] is None: continue
        blk, l = load_block(f"{BASE}/test_series/{uid}/{chosen[p]}")
        if blk is None: continue
        vol[pi] = blk
        if lat is None: lat = l
    if lat == "R":
        vol = vol[:, :, :, ::-1].copy()
    return vol

# ---------- মডেল ----------
class Net(nn.Module):
    def __init__(self, name="efficientnet_b0"):
        super().__init__()
        self.bb = timm.create_model(name, pretrained=False, in_chans=1, num_classes=0)
        self.head = nn.Linear(self.bb.num_features, len(TARGETS))
    def forward(self, x):
        B, P, S, H, W = x.shape
        f = self.bb(x.reshape(B*P*S, 1, H, W)).reshape(B, P*S, -1)
        return self.head(f.mean(1))

wpath = glob.glob("/kaggle/input/**/best-v1.pt", recursive=True)[0]
print("weight:", wpath)

model = Net().to(dev)
model.load_state_dict(torch.load(wpath, map_location=dev))
model.eval()

# ---------- প্রেডিকশন ----------
preds = np.full((len(sub), len(TARGETS)), 0.5, np.float32)   # নিরাপদ ডিফল্ট

for i, uid in enumerate(sub["StudyInstanceUID"].values):
    if time.time() - T0 > TIME_BUDGET:
        print(f"⚠️ সময় শেষ, {i} টায় থামলাম", flush=True); break
    try:
        vol = build_study(uid).astype(np.float32) / 255.
        x = torch.from_numpy(vol)[None].to(dev)
        with torch.no_grad(), torch.autocast("cuda", torch.float16):
            preds[i] = torch.sigmoid(model(x)).float().cpu().numpy()[0]
    except Exception as e:
        print(f"স্টাডি {i} ব্যর্থ: {str(e)[:60]}", flush=True)
    if (i+1) % 100 == 0:
        el = time.time() - T0
        print(f"{i+1}/{len(sub)} | {el/60:.1f} মিনিট | প্রতি স্টাডি {el/(i+1):.2f}s", flush=True)

sub[TARGETS] = preds
sub.to_csv("submission.csv", index=False)

print("\nসম্পন্ন:", sub.shape)
print(sub.head())
print("\nপ্রতি কলামের গড়:")
print(sub[TARGETS].mean())

In [1]:
import glob, os

for f in glob.glob("/kaggle/input/**/*.pt", recursive=True):
    size = os.path.getsize(f) / 1e6
    print(f"{size:6.1f} MB   {f}")

  16.4 MB   /kaggle/input/datasets/prashantadas27/best-epoch/best.pt
  44.8 MB   /kaggle/input/datasets/prashantadas27/knee-model-v2/model_ep1.pt
  44.8 MB   /kaggle/input/datasets/prashantadas27/knee-model-v2/model_ep2.pt
  16.4 MB   /kaggle/input/datasets/prashantadas27/best-version1/best-v1.pt
  16.4 MB   /kaggle/input/datasets/prashantadas27/model-ep6/model_ep6.pt
